# Physical Activity

In [1]:
%pip install -q -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.


  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> [15 lines of output]
      The 'sklearn' PyPI package is deprecated, use 'scikit-learn'
      rather than 'sklearn' for pip commands.
      
      Here is how to fix this error in the main use cases:
      - use 'pip install scikit-learn' rather than 'pip install sklearn'
      - replace 'sklearn' by 'scikit-learn' in your pip requirements files
        (requirements.txt, setup.py, setup.cfg, Pipfile, etc ...)
      - if the 'sklearn' package is used by one of your dependencies,
        it would be great if you take some time to track which package uses
        'sklearn' instead of 'scikit-learn' and report it to their issue tracker
      - as a last resort, set the environment variable
        SKLEARN_ALLOW_DEPRECATED_SKLEARN_PACKAGE_INSTALL=True to avoid this error
      
      More information is available at
      https://github.com/scikit-learn/sklearn-

In [19]:
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import entropy


In [2]:
import pandas as pd
import os

# Load CSV files
data_dir = '../mcphases/'

active_minutes = pd.read_csv(os.path.join(data_dir, 'active_minutes.csv'))
calories = pd.read_csv(os.path.join(data_dir, 'calories.csv'))
demographic_vo2_max = pd.read_csv(os.path.join(data_dir, 'demographic_vo2_max.csv'))
exercise = pd.read_csv(os.path.join(data_dir, 'exercise.csv'))
time_in_heart_rate_zones = pd.read_csv(os.path.join(data_dir, 'time_in_heart_rate_zones.csv'))
height_and_weight = pd.read_csv(os.path.join(data_dir, 'height_and_weight.csv'))
subject_info = pd.read_csv(os.path.join(data_dir, 'subject-info.csv'))
hormones_and_selfreport = pd.read_csv(os.path.join(data_dir, 'hormones_and_selfreport.csv'))

print("All CSV files loaded successfully!")

All CSV files loaded successfully!


In [ ]:
active_minutes.head()

## Daily variation of light -> heavy exercise distribution (phase-specific)

In [ ]:
# Join active_minutes with hormones_and_selfreport on (id, day_in_study)
print(f"Active minutes shape: {active_minutes.shape}")
print(f"Hormones and self-report shape: {hormones_and_selfreport.shape}")

merged_data = active_minutes.merge(
    hormones_and_selfreport,
    on=['id', 'day_in_study'],
    how='inner'
)
merged_data = merged_data.drop(columns=['is_weekend_y'])
merged_data = merged_data.drop(columns=['study_interval_y'])
merged_data = merged_data.rename(
    columns={'is_weekend_x': 'is_weekend'}
)

# Keep only the specified features
features_to_keep = ['id', 'day_in_study', 'is_weekend', 'sedentary', 'lightly', 'moderately', 'very', 'phase']
merged_data = merged_data[features_to_keep]

print(f"Merged data shape: {merged_data.shape}")
print("\nFirst few rows:")
print(merged_data.head())

### Overall

In [ ]:
# stacked bar plot of mean activity by phase
activity = ['sedentary','lightly','moderately','very']

phase_order = [
    'Follicular',
    'Fertility',
    'Luteal',
    'Menstrual'
]

phase_comp = (
    merged_data.groupby('phase')[activity]
    .mean()
)

phase_comp = phase_comp.div(
    phase_comp.sum(axis=1),
    axis=0
)
phase_comp = phase_comp.reindex(phase_order)

phase_comp.plot(kind='bar', stacked=True)

plt.xlabel('Phase')
plt.ylabel('Proportion of Activity')
plt.legend(title='Activity Level')
plt.show()

In [ ]:
# stacked bar plot of mean activity by phase
activity = ['lightly','moderately','very']

phase_order = [
    'Follicular',
    'Fertility',
    'Luteal',
    'Menstrual'
]

phase_comp = (
    merged_data.groupby('phase')[activity]
    .mean()
)

phase_comp = phase_comp.div(
    phase_comp.sum(axis=1),
    axis=0
)
phase_comp = phase_comp.reindex(phase_order)

phase_comp.plot(kind='bar', stacked=True)

plt.xlabel('Phase')
plt.ylabel('Proportion of Activity')
plt.legend(title='Activity Level')
plt.show()

In [ ]:
# violin plots of activity by phase
activity_cols = ['sedentary', 'lightly', 'moderately', 'very']

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for ax, activity in zip(axes.flatten(), activity_cols):
    sns.violinplot(
        data=merged_data,
        x='phase',
        y=activity,
        ax=ax
    )
    ax.set_title(activity.capitalize())

plt.tight_layout()
plt.show()

### Individual

In [ ]:
phase_order = [
    'Follicular',
    'Fertility',
    'Luteal',
    'Menstrual'
]

participant_phase = (
    merged_data
    .groupby(['id', 'phase'])['sedentary']
    .mean()
    .reset_index()
)

# enforce ordering
participant_phase['phase'] = pd.Categorical(
    participant_phase['phase'],
    categories=phase_order,
    ordered=True
)

participant_phase = participant_phase.sort_values('phase')

plt.figure(figsize=(10, 6))

sns.lineplot(
    data=participant_phase,
    x='phase',
    y='sedentary',
    hue='id',
    estimator=None,
    alpha=0.3,
    legend=False
)

plt.title('Participant Mean Sedentary Activity by Phase')
plt.show()

In [ ]:
phase_order = [
    'Follicular',
    'Fertility',
    'Luteal',
    'Menstrual'
]

participant_phase = (
    merged_data
    .groupby(['id', 'phase'])['lightly']
    .mean()
    .reset_index()
)

# enforce ordering
participant_phase['phase'] = pd.Categorical(
    participant_phase['phase'],
    categories=phase_order,
    ordered=True
)

participant_phase = participant_phase.sort_values('phase')

plt.figure(figsize=(10, 6))

sns.lineplot(
    data=participant_phase,
    x='phase',
    y='lightly',
    hue='id',
    estimator=None,
    alpha=0.3,
    legend=False
)

plt.title('Participant Mean Light Activity by Phase')
plt.show()

In [ ]:
phase_order = [
    'Follicular',
    'Fertility',
    'Luteal',
    'Menstrual'
]

participant_phase = (
    merged_data
    .groupby(['id', 'phase'])['moderately']
    .mean()
    .reset_index()
)

# enforce ordering
participant_phase['phase'] = pd.Categorical(
    participant_phase['phase'],
    categories=phase_order,
    ordered=True
)

participant_phase = participant_phase.sort_values('phase')

plt.figure(figsize=(10, 6))

sns.lineplot(
    data=participant_phase,
    x='phase',
    y='moderately',
    hue='id',
    estimator=None,
    alpha=0.3,
    legend=False
)

plt.title('Participant Mean Moderate Activity by Phase')
plt.show()

In [ ]:
phase_order = [
    'Follicular',
    'Fertility',
    'Luteal',
    'Menstrual'
]

participant_phase = (
    merged_data
    .groupby(['id', 'phase'])['very']
    .mean()
    .reset_index()
)

# enforce ordering
participant_phase['phase'] = pd.Categorical(
    participant_phase['phase'],
    categories=phase_order,
    ordered=True
)

participant_phase = participant_phase.sort_values('phase')

plt.figure(figsize=(10, 6))

sns.lineplot(
    data=participant_phase,
    x='phase',
    y='very',
    hue='id',
    estimator=None,
    alpha=0.3,
    legend=False
)

plt.title('Participant Mean Vigorous Activity by Phase')
plt.show()

## Identifying the Target

In [17]:
# Convert self-report symptoms to numeric values
likert_map = {
    'Not at all': 0,
    'Very Low/Little': 1,
    'Low': 2,
    'Moderate': 3,
    'High': 4,
    'Very High': 5
}

symptoms = [
    'appetite',
    'exerciselevel',
    'headaches',
    'cramps',
    'sorebreasts',
    'fatigue',
    'sleepissue',
    'moodswing',
    'stress',
    'foodcravings',
    'indigestion',
    'bloating'
]

for col in symptoms:
    hormones_and_selfreport[col + '_num'] = (
        hormones_and_selfreport[col]
        .map(likert_map)
    )

In [ ]:
# Take a look at the first few rows of the updated dataframe
hormones_and_selfreport.head()

,id,study_interval,is_weekend,day_in_study,phase,lh,estrogen,pdg,flow_volume,flow_color,...,appetite_num,exerciselevel_num,headaches_num,cramps_num,sorebreasts_num,sleepissue_num,stress_num,foodcravings_num,indigestion_num,bloating_num
0,1,2022,True,1,Follicular,2.9,94.2,NaN,Not at all,Not at all,...,2.0,2.0,4.0,1.0,1.0,2.0,3.0,1.0,1.0,1.0
1,1,2022,False,2,Follicular,1.2,226.3,NaN,Not at all,Not at all,...,2.0,2.0,5.0,1.0,1.0,5.0,3.0,1.0,1.0,1.0
2,1,2022,False,3,Follicular,3.5,276.8,NaN,Not at all,Not at all,...,NaN,NaN,4.0,1.0,1.0,5.0,2.0,1.0,1.0,1.0
3,1,2022,False,4,Fertility,1.8,322.1,NaN,Not at all,Not at all,...,2.0,2.0,1.0,1.0,1.0,5.0,2.0,1.0,1.0,1.0
4,1,2022,False,5,Fertility,4.6,244.9,NaN,Not at all,Not at all,...,NaN,NaN,1.0,1.0,1.0,4.0,2.0,1.0,1.0,1.0


In [ ]:
# Analyze the distribution of numeric symptoms and calculate entropy
results = []

for col in symptoms:

    numeric = hormones_and_selfreport[col + '_num']

    probs = (
        numeric
        .value_counts(normalize=True)
        .sort_index()
    )

    results.append({
        'symptom': col,
        'missing_pct': numeric.isna().mean() * 100,
        'std': numeric.std(),
        'entropy': entropy(probs)
    })

results = pd.DataFrame(results)

results.sort_values(
    ['std','entropy'],
    ascending=False
)

,symptom,missing_pct,std,entropy
9,foodcravings,41.208694,1.526374,1.688091
11,bloating,41.191023,1.478250,1.622679
5,fatigue,41.138010,1.476416,1.715616
6,sleepissue,41.173352,1.475101,1.714754
8,stress,41.332391,1.465387,1.700836
7,moodswing,41.332391,1.422916,1.607780
10,indigestion,41.244036,1.416349,1.574793
2,headaches,41.332391,1.405944,1.581243
3,cramps,41.208694,1.268866,1.355679
4,sorebreasts,41.208694,1.113906,1.254055


Excerciselevel and appitite are missing much more data, so exclude those.
Most cycle-related & affected by exercise: fatigue, stress, mood, sleepissue, cramps.
Of the 5, fatigue has the most variability. Thus, fatigue is the ideal target!

In [ ]:
symptoms = [
    'headaches_num',
    'cramps_num',
    'sorebreasts_num',
    'fatigue_num',
    'sleepissue_num',
    'moodswing_num',
    'stress_num',
    'foodcravings_num',
    'bloating_num',
    'Indentation_num'
]

phase_means = (
    hormones_and_selfreport
    .groupby('phase')[symptoms]
    .mean()
)

print(phase_means)

            headaches_num  cramps_num  sorebreasts_num  fatigue_num  \
phase                                                                 
Fertility        1.363388    0.762295         0.665301     2.395095   
Follicular       1.543504    0.641667         0.552381     2.489893   
Luteal           1.318386    0.933751         0.949866     2.469589   
Menstrual        1.710900    1.886970         1.279435     2.695447   

            sleepissue_num  moodswing_num  stress_num  foodcravings_num  \
phase                                                                     
Fertility         2.047814       1.478142    2.549180          1.695355   
Follicular        1.951249       1.474374    2.601907          1.804762   
Luteal            1.980322       1.518386    2.502242          1.820949   
Menstrual         2.113030       1.870458    2.677725          2.023548   

            bloating_num  
phase                     
Fertility       1.517760  
Follicular      1.420927  
Luteal        

Cramps, sorebreasts, and bloating are classical, phase-driven symptoms. This rigid pattern of menstruation--highest leaves little room for excercise to play any role.

Fatigue leaves room for day-to-day variability across the cycle. It also has been well-established that excercise and recovery correlates with fatigue.